In [1]:
import pandas as pd 
import os
import json

# Cleaning and Uploading Dataset

## Functions

### Renaming Files

In [2]:
def rename_files(path, prefix):
    """
    Renames image files and their associated .json and .txt files
    in the given directory using a sequential prefix-based naming scheme.

    Args:
        path (str):   Path to the directory containing the files.
        prefix (str): Prefix name for the renamed files (e.g., 'nightlife').
    """
    img_extensions = ('.jpg', '.png', '.webp', '.jpeg')

    # Get and sort image files for consistent ordering
    img_files = sorted([f for f in os.listdir(path) if f.endswith(img_extensions)])

    for i, img_name in enumerate(img_files, start=1):
        img_ext = os.path.splitext(img_name)[1]  # e.g., '.jpg'

        # Old paths
        old_img  = os.path.join(path, img_name)
        old_json = os.path.join(path, img_name + '.json')
        old_txt  = os.path.join(path, img_name + '.txt')

        # New names
        new_base = f'{prefix}_{i}{img_ext}'      
        new_img  = os.path.join(path, new_base)
        new_json = os.path.join(path, new_base + '.json')
        new_txt  = os.path.join(path, new_base + '.txt')

        if os.path.exists(old_img):
            os.rename(old_img, new_img)
        if os.path.exists(old_json):
            os.rename(old_json, new_json)
        if os.path.exists(old_txt):
            os.rename(old_txt, new_txt)


### Labeling Files (Unsafe/Safe)

Dilakukan di python bukan di ipynb karena cv2 tidak akan muncul di ipynb

In [3]:
def label_files(path, unsafe_path, safe_path):
    """
    Label file berdasarkan safe/unsafe lalu simpan ke dalam 
    sebuah folder yang berbeda.

    Args:
        path (str): File path asli.
        unsafe_path (str): File path untuk menempatkan unsafe images.
        safe_path(str): File path untuk menempatkan safe images.
    
    Controls:
        [u] = Unsafe
        [s] = Safe
        [q] = Quit
    """

    # Buat kedua folder jika tidak ada
    os.makedirs(unsafe_path, exist_ok=True)  
    os.makedirs(safe_path, exist_ok=True)    

    for filename in os.listdir(path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.webp')):
            img_path = os.path.join(path, filename)
            img = cv2.imread(img_path)

            display_img = cv2.resize(img, (500, 500))
            cv2.imshow('Labeling - [u] Unsafe  [s] Safe  [q] Quit', display_img)

            key = cv2.waitKey(0)

            if key == ord('u'):
                shutil.move(img_path, os.path.join(unsafe_path, filename))
                print(f"[UNSAFE] {filename}")
            elif key == ord('s'):
                shutil.move(img_path, os.path.join(safe_path, filename)) 
                print(f"[SAFE]   {filename}")
            elif key == ord('q'):
                print("Quit labeling.")
                break

    cv2.destroyAllWindows()

### Creating DataFrame

In [4]:
def build_dataframe(tag_name, violation_type):
    """
    Membuat dataframe dari image yang sudah dilabeli agar sesuai dengan dataframe di huggingface

    Keterangan:
        sample_name    - nama file
        description    - untuk diisi image captioning
        category       - 'safe' atau 'unsafe'
        violation_type - jenis pelanggaran
        type           - 'image'
        link           - post url dari json
        image          - gambar

    Args:
        tag_name (str):       dataset folder
        violation_type (str): violation type

    Returns:
        pd.DataFrame
    """
    img_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    records = []

    for category in ['safe', 'unsafe']:
        folder_path = os.path.join(tag_name, category)

        if not os.path.exists(folder_path):
            print(f"Folder not found, skipping: {folder_path}")
            continue

        for filename in sorted(os.listdir(folder_path)):
            if not filename.endswith(img_extensions):
                continue

            img_path  = os.path.join(folder_path, filename)

            # Look for the matching .json sidecar file (same folder as image)
            # Original scrape keeps .json next to the images in gallery-dl output,
            # but after labeling the images move — so check current folder first.
            base_path = 'gallery-dl/instagram/tag'
            json_path = os.path.join(base_path, tag_name, filename) + '.json'
            post_url  = None

            if os.path.exists(json_path):
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    post_url = data.get('post_url', None)

            records.append({
                'sample_name': filename,
                'description': None,
                'category': category,
                'violation_type': violation_type,
                'type': 'image',
                'link': post_url,
                'image': img_path
            })

    df = pd.DataFrame(records)
    return df


### Upload to HF

In [5]:
from datasets import Dataset,Image, concatenate_datasets, load_dataset
from huggingface_hub import login

In [6]:
from datasets import Dataset, Image
from huggingface_hub import login

def upload_to_huggingface(df, repo_name, hf_token):
    """
    Convert DataFrame to Datasets

    Args:
        df         (pd.DataFrame): DataFrame from build_dataset()
        repo_name  (str):          e.g. 'your-username/your-dataset-name'
        hf_token   (str):          Your HuggingFace token (Write permission)
    """
    login(token=hf_token)

    new_dataset = Dataset.from_pandas(df, preserve_index=False)
    new_dataset = new_dataset.cast_column("image", Image())

    existing_dataset = load_dataset(repo_name, split='train')
    combined_dataset = concatenate_dataset([existing_dataset, new_dataset])

    combined_dataset.push_to_hub(repo_name, split="train", append=True)


### Path to Image (another HF approach)

In [32]:
from PIL import Image
import io

In [34]:
def path_to_bytes(img_path):
    """Baca path dan convert ke bytes"""
    try:
        with Image.open(img_path) as img:
            buf = io.BytesIO()
            img.save(buf, format=img.format or "WEBP")
            return buf.getvalue()
    except Exception as e:
        print(f"Failed: {img_path} → {e}")
        return None

## Implementation

In [7]:
import uuid
from dotenv import load_dotenv
HF_TOKEN = os.getenv("HF_TOKEN")

### indoclubbing

In [35]:
df_indoclubbing = build_dataframe('indoclubbing', 'adult_lifestyle')

In [37]:
df_indoclubbing["image"] = df_indoclubbing["image"].apply(path_to_bytes)
shard_name = f"train-{uuid.uuid4().hex[:8]}.parquet"
df_indoclubbing.to_parquet(shard_name)

### jomok

In [6]:
rename_files('gallery-dl/instagram/tag/jomok', 'jomok')

In [39]:
df_jomok = build_dataframe('jomok', 'inappropriate_humor')

In [41]:
df_jomok["image"] = df_jomok["image"].apply(path_to_bytes)
shard_name = f"train-{uuid.uuid4().hex[:8]}.parquet"
df_jomok.to_parquet(shard_name)

### anakdugem

In [ ]:
rename_files('anakdugem/safe', 'anakdugem')